# Toy models: the latent sphere of each objective

Shows where every pixel lands on the latent sphere $S^2$ for each objective and how the arrangement changes when a loss term is added. It answers whether the reconstruction-only latent already separates the annotated ions and what the head, contrastive and contractive terms change.

## Introduction

The toy campaign (`kidney-representative-toy`) trains cumulative objectives on one small kidney image with latent width $L = 4$, so that the whole latent space is $S^2$ and can be shown directly. Its figures are used in the conceptual presentation (`assets/documents/prestentations/segmentation_model_full_conceptual`).

### Assumptions

- One model per objective (one repetition). Differences between variants are single realizations, not
  estimates of an expected effect; the toy is illustrative, not a benchmark.
- All variants share the data split, the initialization seed and the training seed, so the only
  difference is the objective.
- Architecture, optimizer, loss parameters and loss weights follow the `real_only` schedule of the
  `20_09_26_metaspace_base_pretrain` campaign (without label-weighted contrastive negatives), including
  its 10 epochs; only `latent_dim` is smaller.
- Ten epochs on the training part of 16 230 pixels are about 2000 optimizer steps; the
  auxiliary terms (weights 0.2, 0.001, 0.002) contribute little to the objective, so differences between
  variants are expected to be small.
- Display categories are built from five selected ions (three spatially disjoint, one overlapping one of
  them, one present almost everywhere) by the rules in `analysis_settings.yaml`; names list the ions.

### Notation

| symbol | meaning |
|---|---|
| $x \in \Delta^{M-1}$ | TIC-normalized binned spectrum, $M = 1364$ bins of $0.55$ m/z on $[200, 950]$ |
| $a \in \mathbb{R}^{L}$ | encoder output before the bottleneck `LayerNorm`, $L = 4$ |
| $u = (a - \mu_a \mathbf 1)/\sigma_a$ | canonical latent, $\mathbf 1^\top u = 0$, $\lVert u \rVert = \sqrt L$ |
| $z = \gamma \odot u + \beta$ | model latent (affine `LayerNorm` output) |
| $s = Q^\top u / \lVert Q^\top u \rVert \in S^2$ | latent coordinates; $Q \in \mathbb{R}^{4 \times 3}$ fixed orthonormal basis of $\mathbf 1^\perp$ |
| $\mathcal M_0, \dots, \mathcal M_4$ | objectives: reconstruction; + head; + head + contrastive; + head + contractive; + head + both |
| $\angle(u, u')$ | angle between canonical latents, in degrees |

## Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import representative_toy_precompute as toy
from msi_autoencoder_wrapper.visualization import representative_toy as viz

SETTINGS = toy.load_settings(Path("analysis_settings.yaml"))
FIGURES = Path(SETTINGS["repository_root"]) / SETTINGS["figures"]["presentation_directory"]
LABELS = SETTINGS["campaign"]["labels"]
SHOWN = SETTINGS["campaign"]["presentation"]  # variants used on the slides
PREFIX = SETTINGS["figures"]["prefix"]
REGION_COLORS = {rule["name"]: rule["color"] for rule in SETTINGS["regions"]}

classes = toy.load_table(SETTINGS, "dataset", "classes")
pixels = toy.load_table(SETTINGS, "dataset", "pixels")
pixels["region"] = toy.assign_regions(pixels, classes, SETTINGS["regions"])

## Producing the tables this notebook reads

The tables are written by the toy precomputation module; the command below is generated by that module.

In [ ]:
print(toy.run_command(SETTINGS, "latent"))

## Canonical latent lies on the sphere

### Methodology

#### Theoretical

For $L = 4$ the canonical latent satisfies $\mathbf 1^\top u = 0$ and $\lVert u \rVert^2 = L\sigma^2/(\sigma^2+\varepsilon)$, so it lies on $S^{2}(\sqrt 4)$ inside the three-dimensional subspace $\mathbf 1^\perp$.

#### Implementation

`u = (z - beta) / gamma` with the bottleneck `LayerNorm` parameters of each model (`sphere_geometry.canonicalize`); `verify_canonicalization` reports row sums and norms.

#### Figure descriptions

No figure; one row per variant with the canonicalization diagnostics.

In [ ]:
latent = toy.load_table(SETTINGS, "latent", "latent").merge(pixels[["source_id", "region"]], on="source_id")
display(toy.load_table(SETTINGS, "latent", "canonicalization"))
display(latent.groupby("variant", sort=False).masserstein.describe()[["mean", "50%", "max"]])

### Remarks

### Notes

## Latent sphere and image per objective

### Methodology

#### Theoretical

The chart $s = Q^\top u/\lVert Q^\top u\rVert$ is an isometry of the latent sphere onto the unit sphere of $\mathbb R^3$. Consecutive variants are aligned by an orthogonal Procrustes rotation to the previous variant; rotations preserve every angle, so only the orientation of the view is normalized.

#### Implementation

Sphere coordinates and rotations are computed in the precomputation (`latent` analysis) for all 16 230 pixels (every 3rd pixel of each section is drawn on the sphere). The image colour maps each sphere coordinate linearly from its range over the compared variants to $[0.1, 0.9]$ (shared by the panels of one figure), so pixels close on the sphere have similar colours. The camera looks at the mean latent direction of the compared variants.

#### Figure descriptions

Each column is one variant. Top: unit sphere, one point per displayed pixel, colour = display category, points on the far hemisphere are faded. Bottom: the toy image in the same category colours.

In [ ]:
frames = {name: frame for name, frame in latent.groupby("variant", sort=False)}
pairs = {"sphere_m0": ("m0-reconstruction",), "sphere_m0_vs_m1": ("m0-reconstruction", "m1-head"),
         "sphere_m1_vs_contrastive": ("m1-head", SHOWN["contrastive"]),
         "sphere_m1_vs_contractive": ("m1-head", SHOWN["contractive"]),
         "sphere_m1_vs_both": ("m1-head", SHOWN["both"])}
with viz.presentation_style():
    for name, pair in pairs.items():
        figure = viz.plot_variant_pair([(LABELS[v], frames[v]) for v in pair], REGION_COLORS,
                                       background=SETTINGS["background_regions"])
        viz.save_figure(figure, FIGURES, PREFIX + name)
        plt.show()

In [ ]:
shown = ["m0-reconstruction", "m1-head", SHOWN["contrastive"], SHOWN["contractive"], SHOWN["both"]]
if "sphere_c" in latent:  # the interactive view exists only for the two-sphere
    viz.plotly_sphere_html({LABELS[name]: frames[name] for name in shown},
                           FIGURES / (PREFIX + "sphere_interactive.html"), color_column="region",
                           color_map=REGION_COLORS, hover_columns=("region", "source_id"), title="", show_legend=False, display_column="display")

### Remarks

### Notes

## Annotated ions on the sphere without and with the head

### Methodology

#### Theoretical

For a marker ion $c$, the pixels with $y_{pc} = 1$ are highlighted on the sphere. Without the head ($\mathcal M_0$) the latent is shaped only by reconstruction; with the head ($\mathcal M_1$) the latent is additionally shaped by a classifier $h(z)$ trained on the annotations with the variational PU loss.

#### Implementation

Same coordinates as above; ion presence is the training target of the dataset.

#### Figure descriptions

Rows = marker ions (m/z), columns = variants; red = ion annotated in the pixel, grey = not annotated.

In [ ]:
markers = [909.558, 826.576, 840.555, 723.508]
ions = [(f"{mz:.1f}", "label::" + classes.loc[(classes.mz - mz).abs().idxmin(), "class_name"]) for mz in markers]
with viz.presentation_style():
    figure = viz.plot_ion_presence_spheres([(LABELS[n], frames[n]) for n in ("m0-reconstruction", "m1-head")], ions)
    viz.save_figure(figure, FIGURES, PREFIX + "head_ions")
    plt.show()

### Remarks

### Notes

## Results / Summary

### LLM

### Person